# MMFi Leave-One-Out Results Analysis

Simple notebook to aggregate and visualize leave-one-out training results.
Reads `best_metrics.json` directly from checkpoint directories.


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from pathlib import Path
from cycler import cycler

# Checkpoint directory
CHECKPOINT_DIR = Path("../../../logs/pose_estimation/mmfi/leave_one_out/checkpoints")

# ==============================
# Paper Figure Style Configuration
# ==============================
def get_transparent_color(color, transparency=0.5):
    """Convert a hex color to a lighter (more transparent) version."""
    c = mcolors.hex2color(color)
    c = [c[0] * transparency + (1.0 - transparency),
         c[1] * transparency + (1.0 - transparency),
         c[2] * transparency + (1.0 - transparency)]
    return "#{:02X}{:02X}{:02X}".format(int(c[0]*255), int(c[1]*255), int(c[2]*255))

palette = ['#1e90ff', '#ffbb00', '#ff5080', '#a7426d', '#ff3c10', "#282828"]
palette_facecolor = [get_transparent_color(c) for c in palette]
markers = ['o', 'P', '^', 's', 'p', 'h']
hatches = ['//', '\\\\', 'xx', '///', '\\\\\\', 'xxx']

plt.rcdefaults()
plt.rcParams["figure.figsize"] = [8, 4]
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 300

plt.rcParams["grid.linestyle"] = "--"
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["ytick.direction"] = "in"

plt.rcParams['lines.linewidth'] = 2.5
plt.rcParams['lines.markersize'] = 8
plt.rcParams['lines.markeredgewidth'] = 2.0

plt.rcParams["font.size"] = 18
plt.rcParams["font.family"] = "Arial"
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

plt.rcParams["legend.fontsize"] = "medium"
plt.rcParams["legend.facecolor"] = "white"
plt.rcParams["legend.edgecolor"] = "white"
plt.rcParams["legend.framealpha"] = 0.9
plt.rcParams['legend.frameon'] = False
plt.rcParams['legend.handlelength'] = 1.5
plt.rcParams['legend.handletextpad'] = 0.5
plt.rcParams['legend.columnspacing'] = 0.8
plt.rcParams['legend.labelspacing'] = 0.3

plt.rcParams["axes.prop_cycle"] = cycler(color=palette) + cycler(marker=markers)


In [ ]:
# Load all best_metrics.json files
results = {}

for subdir in CHECKPOINT_DIR.iterdir():
    if not subdir.is_dir():
        continue
    metrics_file = subdir / "best_metrics.json"
    if metrics_file.exists():
        with open(metrics_file) as f:
            metrics = json.load(f)
        exp_name = metrics.get("experiment_name", subdir.name)
        results[exp_name] = metrics

print(f"Found {len(results)} experiments")
print(f"Example keys: {list(results.keys())[:3]}")

# Parse results into leave-one-out experiments
# (Baseline/All-included results come from evaluate_mmfi_pretrained.py output)
loo_results = {}  # action -> [mpjpe values]

for exp_name, metrics in results.items():
    test_mpjpe = metrics.get("test", {}).get("mpjpe")
    if test_mpjpe is None:
        continue
    
    if exp_name.startswith("leave_out"):
        # Extract action: leave_out_A01_repeat00 -> A01
        parts = exp_name.split("_")
        for p in parts:
            if p.startswith("A") and len(p) == 3:
                action = p
                if action not in loo_results:
                    loo_results[action] = []
                loo_results[action].append(test_mpjpe)
                break

print(f"Leave-one-out actions: {len(loo_results)}")


In [ ]:
# Prepare data for plotting
actions_sorted = sorted(loo_results.keys())
loo_mpjpes = [np.mean(loo_results[a]) for a in actions_sorted]

# Load pretrained model results as baseline (from evaluate_mmfi_pretrained.py output)
PRETRAINED_RESULTS = Path("mmfi_pretrained_evaluation.json")
pretrained_action_data = {}

if PRETRAINED_RESULTS.exists():
    with open(PRETRAINED_RESULTS) as f:
        pretrained = json.load(f)
    # Build dict: action -> mpjpe_mean
    for r in pretrained.get("actions", []):
        pretrained_action_data[r["action"]] = r["mpjpe_mean"]
    print(f"Loaded pretrained results for {len(pretrained_action_data)} actions")
else:
    print(f"⚠️ Pretrained results not found: {PRETRAINED_RESULTS}")
    print("   Run evaluate_mmfi_pretrained.py first to generate baseline data")

# Get pretrained mpjpe for each action (aligned with loo_results)
pretrained_mpjpes = [pretrained_action_data.get(a, None) for a in actions_sorted]
pretrained_mean = np.mean([v for v in pretrained_mpjpes if v is not None]) if pretrained_action_data else None

print("\n=== Results Summary ===")
if pretrained_mean:
    print(f"Pretrained (All-included) Mean MPJPE: {pretrained_mean:.2f} mm")
print(f"Leave-one-out Mean: {np.mean(loo_mpjpes):.2f} ± {np.std(loo_mpjpes):.2f} mm")
print(f"Leave-one-out Range: {np.min(loo_mpjpes):.2f} - {np.max(loo_mpjpes):.2f} mm")


In [ ]:
# Plot: Leave-one-out vs All-included (Paper Figure Style)
lw = 2

fig, ax = plt.subplots(figsize=(10, 5))

positions = np.arange(len(loo_mpjpes))

# Bar chart - Leave-one-out training results
bars = ax.bar(positions, loo_mpjpes, width=0.8,
              facecolor=palette_facecolor[0], edgecolor=palette[0],
              linewidth=lw, hatch=hatches[0], alpha=0.8,
              label='Leave-one-out')

# Line plot - Pretrained (All-included) model results
if pretrained_action_data:
    pretrained_plot_data = [pretrained_action_data.get(a, np.nan) for a in actions_sorted]
    ax.plot(positions, pretrained_plot_data,
            color=palette[1], marker=markers[1], markersize=6,
            linewidth=lw+1, markeredgewidth=2,
            label='All-included', linestyle='-')

ax.set_xticks(positions[::3])
ax.set_xticklabels([actions_sorted[i] for i in range(0, len(actions_sorted), 3)])
ax.set_xlabel("Action")
ax.set_ylabel("MPJPE (mm)")
ax.set_ylim(0, 200)
ax.grid(True, axis='y')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.2), ncol=2, frameon=False)
ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=5))

plt.tight_layout()
plt.subplots_adjust(top=0.85)
plt.savefig('leave_one_out_vs_all_included_action.pdf', bbox_inches='tight', dpi=300)
plt.show()

# Save aggregated results
output = {
    "pretrained": pretrained_action_data,  # All-included baseline per action
    "leave_one_out": {action: loo_results[action] for action in actions_sorted},
    "summary": {
        "pretrained_mean_mpjpe": float(pretrained_mean) if pretrained_mean else None,
        "loo_mean_mpjpe": float(np.mean(loo_mpjpes)),
        "loo_std_mpjpe": float(np.std(loo_mpjpes)),
    }
}

output_path = CHECKPOINT_DIR.parent / "leave_one_out_summary.json"
with open(output_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"✅ Results saved to: {output_path}")
